<a href="https://colab.research.google.com/github/KrasnovaOV/Big-Data-Analytics/blob/main/%D0%98%D1%82%D0%BE%D0%B3%D0%BE%D0%B2%D0%BE%D0%B5_%D0%90%D0%91%D0%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практика Python**
В данном разделе мы выступим в роли data scientist и попытаемся построить простую модель для рекомендации фильмов пользователям.

Задание

1.	Загрузите в колаб файлы по оценкам (ratings) и фильмам (movies) и создайте на их основе pandas-датафреймы

Сформировав общий топ фильмов в прошлой практике, мы хотим сделать шаг вперед и начать советовать пользователю те фильмы, которые могли бы быть для него наиболее интересны. Наша цель - научится предсказывать оценку фильма пользователем. Для тестирования модели найдем пользователя, который поставил больше всего оценок

2.	Средствами Pandas, используя dataframe ratings, найдите id пользователя, поставившего больше всего оценок

Отберем фильмы, которые оценил данный пользователь

3.	Оставьте в датафрейме ratings только те фильмы, который оценил данный пользователь

Для построения модели нам нужны признаки. В качестве таковых будем использовать:

●	Год выхода

●	Жанры

●	Общее количество оценок

●	Суммарную оценку


4.	Добавьте к датафрейму из задания 3 столбцы:

●	По жанрам. Каждый столбец - это жанр. Единицу записываем, если фильм принадлежит данному жанру и 0 - если нет

●	столбцы с общим количеством оценок от всех пользователей на фильм и суммарной оценкой от всех пользователей


Теперь все готово и можно строить модель

5.	Сформируйте X_train, X_test, y_train, y_test
6.	Возьмите модель линейной регрессии (или любую другую для задачи регрессии)  и обучите ее на фильмах
7.	Оцените качество модели на X_test, y_test при помощи метрик для задачи регрессии

Вторая часть практики Python связана со Spark’ом
8.	Загрузить данные в spark
9.	Средствами спарка вывести среднюю оценку для каждого фильма	20
10.	Посчитайте средствами спарка среднюю оценку для каждого жанра	20
11.	В спарке получить 2 датафрейма с 5-ю самыми популярными и самыми непопулярными фильмами (по количеству оценок, либо по самой оценке - на Ваш выбор)




**1. Загрузка данных**

In [1]:
from google.colab import files
uploaded = files.upload()

Saving u.data.csv to u.data.csv
Saving u.item.csv to u.item.csv
Saving u.user.csv to u.user.csv


In [5]:
import os

print(os.listdir())

['.config', 'u.data.csv', 'u.user.csv', 'u.item.csv', 'sample_data']


In [6]:
import pandas as pd

ratings = pd.read_csv(
    "u.data.csv",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
    encoding="latin-1"
)

movie_columns = [
    "item_id", "title", "release_date", "video_release_date", "IMDb_URL",
    "unknown", "Action", "Adventure", "Animation", "Children's",
    "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
    "Sci-Fi", "Thriller", "War", "Western"
]

movies = pd.read_csv(
    "u.item.csv",
    sep="|",
    names=movie_columns,
    encoding="latin-1"
)

print("ratings:", ratings.shape)
print("movies:", movies.shape)

ratings.head(), movies.head()

ratings: (100000, 4)
movies: (1682, 24)


(   user_id  item_id  rating  timestamp
 0      196      242       3  881250949
 1      186      302       3  891717742
 2       22      377       1  878887116
 3      244       51       2  880606923
 4      166      346       1  886397596,
    item_id              title release_date  video_release_date  \
 0        1   Toy Story (1995)  01-Jan-1995                 NaN   
 1        2   GoldenEye (1995)  01-Jan-1995                 NaN   
 2        3  Four Rooms (1995)  01-Jan-1995                 NaN   
 3        4  Get Shorty (1995)  01-Jan-1995                 NaN   
 4        5     Copycat (1995)  01-Jan-1995                 NaN   
 
                                             IMDb_URL  unknown  Action  \
 0  http://us.imdb.com/M/title-exact?Toy%20Story%2...        0       0   
 1  http://us.imdb.com/M/title-exact?GoldenEye%20(...        0       1   
 2  http://us.imdb.com/M/title-exact?Four%20Rooms%...        0       0   
 3  http://us.imdb.com/M/title-exact?Get%20Shorty%...      

Размеры, типы, пропуски, дубликаты, диапазоны значений и базовые проверки корректности данных.

In [7]:
def basic_eda(df: pd.DataFrame, name: str, key_columns=None):
    print("=" * 70)
    print(f"БАЗОВАЯ ПРОВЕРКА: {name}")
    print("=" * 70)

    print(f"\nРазмер датафрейма: {df.shape[0]} строк, {df.shape[1]} столбцов")

    print("\nСтруктура и типы данных:")
    print(df.info())

    print("\nПервые 5 строк:")
    display(df.head())

    print("\nПропуски по столбцам:")
    missing = df.isna().sum()
    missing_percent = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        "missing_count": missing,
        "missing_percent": missing_percent
    })
    display(missing_df[missing_df["missing_count"] > 0]
            if missing_df["missing_count"].sum() > 0
            else missing_df)

    print("\nКоличество полностью дублирующихся строк:")
    print(df.duplicated().sum())

    if key_columns:
        print(f"\nДубликаты по ключу(ам) {key_columns}:")
        print(df.duplicated(subset=key_columns).sum())

    print("\nОписательная статистика числовых столбцов:")
    display(df.describe(include=[np.number]).T)

    print("\nУникальные значения по столбцам:")
    for column in df.columns:
        unique_count = df[column].nunique(dropna=False)
        print(f"{column}: {unique_count} уникальных значений")

    print("\n" + "=" * 70 + "\n")


import numpy as np

basic_eda(
    ratings,
    "ratings",
    key_columns=["user_id", "item_id"]
)

basic_eda(
    movies,
    "movies",
    key_columns=["item_id"]
)

БАЗОВАЯ ПРОВЕРКА: ratings

Размер датафрейма: 100000 строк, 4 столбцов

Структура и типы данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    100000 non-null  int64
 1   item_id    100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB
None

Первые 5 строк:


,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596



Пропуски по столбцам:


,missing_count,missing_percent
user_id,0,0.0
item_id,0,0.0
rating,0,0.0
timestamp,0,0.0



Количество полностью дублирующихся строк:
0

Дубликаты по ключу(ам) ['user_id', 'item_id']:
0

Описательная статистика числовых столбцов:


,count,mean,std,min,25%,50%,75%,max
user_id,100000.0,4.624848e+02,2.666144e+02,1.0,254.0,447.0,682.0,943.0
item_id,100000.0,4.255301e+02,3.307984e+02,1.0,175.0,322.0,631.0,1682.0
rating,100000.0,3.529860e+00,1.125674e+00,1.0,3.0,4.0,4.0,5.0
timestamp,100000.0,8.835289e+08,5.343856e+06,874724710.0,879448709.5,882826944.0,888259984.0,893286638.0


... (3 outputs left out)

,missing_count,missing_percent
release_date,1,0.06
video_release_date,1682,100.00
IMDb_URL,3,0.18



Количество полностью дублирующихся строк:
0

Дубликаты по ключу(ам) ['item_id']:
0

Описательная статистика числовых столбцов:


,count,mean,std,min,25%,50%,75%,max
item_id,1682.0,841.500000,485.695893,1.0,421.25,841.5,1261.75,1682.0
video_release_date,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unknown,1682.0,0.001189,0.034473,0.0,0.00,0.0,0.00,1.0
Action,1682.0,0.149227,0.356418,0.0,0.00,0.0,0.00,1.0
Adventure,1682.0,0.080262,0.271779,0.0,0.00,0.0,0.00,1.0
Animation,1682.0,0.024970,0.156081,0.0,0.00,0.0,0.00,1.0
Children's,1682.0,0.072533,0.259445,0.0,0.00,0.0,0.00,1.0
Comedy,1682.0,0.300238,0.458498,0.0,0.00,0.0,1.00,1.0
Crime,1682.0,0.064804,0.246253,0.0,0.00,0.0,0.00,1.0
Documentary,1682.0,0.029727,0.169882,0.0,0.00,0.0,0.00,1.0



Уникальные значения по столбцам:
item_id: 1682 уникальных значений
title: 1664 уникальных значений
release_date: 241 уникальных значений
video_release_date: 1 уникальных значений
IMDb_URL: 1661 уникальных значений
unknown: 2 уникальных значений
Action: 2 уникальных значений
Adventure: 2 уникальных значений
Animation: 2 уникальных значений
Children's: 2 уникальных значений
Comedy: 2 уникальных значений
Crime: 2 уникальных значений
Documentary: 2 уникальных значений
Drama: 2 уникальных значений
Fantasy: 2 уникальных значений
Film-Noir: 2 уникальных значений
Horror: 2 уникальных значений
Musical: 2 уникальных значений
Mystery: 2 уникальных значений
Romance: 2 уникальных значений
Sci-Fi: 2 уникальных значений
Thriller: 2 уникальных значений
War: 2 уникальных значений
Western: 2 уникальных значений




In [9]:
import numpy as np

genre_columns = movie_columns[5:]

print("Проверка ratings")
print("Диапазон user_id:", ratings["user_id"].min(), "-", ratings["user_id"].max())
print("Диапазон item_id:", ratings["item_id"].min(), "-", ratings["item_id"].max())
print("Диапазон rating:", ratings["rating"].min(), "-", ratings["rating"].max())
print("Уникальные оценки:", sorted(ratings["rating"].unique()))

print("\nПроверка movies")
print("Диапазон item_id:", movies["item_id"].min(), "-", movies["item_id"].max())
print("Количество уникальных фильмов:", movies["item_id"].nunique())

print("\nЖанры: значения должны быть только 0 или 1")
genre_check = movies[genre_columns].isin([0, 1]).all()
display(genre_check)

print("\nФильмы без указанного жанра:")
no_genre = movies[genre_columns].sum(axis=1) == 0
print(no_genre.sum())

print("\nФильмы с несколькими жанрами:")
multiple_genres = movies[genre_columns].sum(axis=1) > 1
print(multiple_genres.sum())

Проверка ratings
Диапазон user_id: 1 - 943
Диапазон item_id: 1 - 1682
Диапазон rating: 1 - 5
Уникальные оценки: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Проверка movies
Диапазон item_id: 1 - 1682
Количество уникальных фильмов: 1682

Жанры: значения должны быть только 0 или 1


,0
unknown,True
Action,True
Adventure,True
Animation,True
Children's,True
Comedy,True
Crime,True
Documentary,True
Drama,True
Fantasy,True



Фильмы без указанного жанра:
0

Фильмы с несколькими жанрами:
849


Дополнительные проверки

Для MovieLens особенно полезно проверить допустимые значения оценок, ID и жанров.

In [10]:
print("Проверка ratings")
print("Диапазон user_id:", ratings["user_id"].min(), "-", ratings["user_id"].max())
print("Диапазон item_id:", ratings["item_id"].min(), "-", ratings["item_id"].max())
print("Диапазон rating:", ratings["rating"].min(), "-", ratings["rating"].max())
print("Уникальные оценки:", sorted(ratings["rating"].unique()))

print("\nПроверка movies")
print("Диапазон item_id:", movies["item_id"].min(), "-", movies["item_id"].max())
print("Количество уникальных фильмов:", movies["item_id"].nunique())

print("\nЖанры: значения должны быть только 0 или 1")
genre_check = movies[genre_columns].isin([0, 1]).all()
display(genre_check)

print("\nФильмы без указанного жанра:")
no_genre = movies[genre_columns].sum(axis=1) == 0
print(no_genre.sum())

print("\nФильмы с несколькими жанрами:")
multiple_genres = movies[genre_columns].sum(axis=1) > 1
print(multiple_genres.sum())

Проверка ratings
Диапазон user_id: 1 - 943
Диапазон item_id: 1 - 1682
Диапазон rating: 1 - 5
Уникальные оценки: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Проверка movies
Диапазон item_id: 1 - 1682
Количество уникальных фильмов: 1682

Жанры: значения должны быть только 0 или 1


,0
unknown,True
Action,True
Adventure,True
Animation,True
Children's,True
Comedy,True
Crime,True
Documentary,True
Drama,True
Fantasy,True



Фильмы без указанного жанра:
0

Фильмы с несколькими жанрами:
849


Проверка соответствия таблиц

In [11]:
print("Оценки по фильмам, которых нет в movies:")
print((~ratings["item_id"].isin(movies["item_id"])).sum())

print("\nФильмы без оценок:")
print((~movies["item_id"].isin(ratings["item_id"])).sum())

print("\nКоличество уникальных пользователей в ratings:")
print(ratings["user_id"].nunique())

print("\nКоличество уникальных фильмов в ratings:")
print(ratings["item_id"].nunique())

Оценки по фильмам, которых нет в movies:
0

Фильмы без оценок:
0

Количество уникальных пользователей в ratings:
943

Количество уникальных фильмов в ratings:
1682


**2. Самый активный пользователь**

In [12]:
# Количество оценок у каждого пользователя
user_counts = (
    ratings
    .groupby("user_id")["rating"]
    .count()
    .sort_values(ascending=False)
)

top_user_id = user_counts.index[0]
top_user_count = user_counts.iloc[0]

print(f"Пользователь с наибольшим числом оценок: {top_user_id}")
print(f"Количество его оценок: {top_user_count}")

# Фильмы, которые оценил этот пользователь
top_user_ratings = ratings[ratings["user_id"] == top_user_id].copy()
top_user_ratings.head()

Пользователь с наибольшим числом оценок: 405
Количество его оценок: 737


,user_id,item_id,rating,timestamp
12276,405,56,4,885544911
12383,405,592,1,885548670
12430,405,1582,1,885548670
12449,405,171,1,885549544
12460,405,580,1,885547447


**3. Оценки только этого пользователя**

In [13]:
ratings_top_user = ratings[ratings["user_id"] == top_user_id].copy()

print(f"Размер исходного ratings: {ratings.shape}")
print(f"Размер ratings для пользователя {top_user_id}: {ratings_top_user.shape}")

ratings_top_user.head()

Размер исходного ratings: (100000, 4)
Размер ratings для пользователя 405: (737, 4)


,user_id,item_id,rating,timestamp
12276,405,56,4,885544911
12383,405,592,1,885548670
12430,405,1582,1,885548670
12449,405,171,1,885549544
12460,405,580,1,885547447


**4. Признаки фильмов**

Добавляем год выпуска, жанры, общее число оценок и суммарную оценку по каждому фильму.

In [14]:
# Год выпуска
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    format="%d-%b-%Y",
    errors="coerce"
)
movies["year"] = movies["release_date"].dt.year

# Список жанров
genre_columns = movie_columns[5:]

# Агрегаты по всем пользователям
movie_stats = (
    ratings
    .groupby("item_id")
    .agg(
        total_ratings=("rating", "count"),
        total_rating_sum=("rating", "sum")
    )
    .reset_index()
)

# Соединяем оценки пользователя с информацией о фильмах
data = (
    ratings_top_user
    .merge(
        movies[["item_id", "year"] + genre_columns],
        on="item_id",
        how="left"
    )
    .merge(movie_stats, on="item_id", how="left")
)

# Если у фильма нет года, заполняем медианой
data["year"] = data["year"].fillna(data["year"].median())

data.head()

,user_id,item_id,rating,timestamp,year,unknown,Action,Adventure,Animation,Children's,...,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,total_ratings,total_rating_sum
0,405,56,4,885544911,1994.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,394,1600
1,405,592,1,885548670,1995.0,0,0,0,0,0,...,0,0,1,0,0,1,0,0,9,30
2,405,1582,1,885548670,1947.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
3,405,171,1,885549544,1991.0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,65,252
4,405,580,1,885547447,1995.0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,32,108


**5. Train и test**

Целевая переменная — оценка rating, которую пользователь поставил фильму. Признаки — год, жанры, общее количество оценок и суммарная оценка.

In [15]:
from sklearn.model_selection import train_test_split

feature_columns = ["year", "total_ratings", "total_rating_sum"] + genre_columns

X = data[feature_columns]
y = data["rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (589, 22)
X_test: (148, 22)
y_train: (589,)
y_test: (148,)


**6. Обучение модели**

In [16]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

**7. Оценка качества**

Используем MAE, MSE, RMSE и
𝑅
2
R
2
 . mean_squared_error вычисляет среднеквадратичную ошибку, а r2_score — коэффициент детерминации.

In [17]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.3f}")
print(f"MSE:  {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R^2:  {r2:.3f}")

MAE:  0.951
MSE:  1.699
RMSE: 1.304
R^2:  0.156


**8. Загрузка в Spark**

In [18]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as spark_sum, desc, asc

spark = SparkSession.builder.appName("MovieLens100K").getOrCreate()

ratings_spark = spark.createDataFrame(ratings)
movies_spark = spark.createDataFrame(movies)

ratings_spark.printSchema()
movies_spark.printSchema()

ratings_spark.show(5)
movies_spark.show(5, truncate=False)

root
 |-- user_id: long (nullable = true)
 |-- item_id: long (nullable = true)
 |-- rating: long (nullable = true)
 |-- timestamp: long (nullable = true)

root
 |-- item_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- release_date: timestamp (nullable = true)
 |-- video_release_date: double (nullable = true)
 |-- IMDb_URL: string (nullable = true)
 |-- unknown: long (nullable = true)
 |-- Action: long (nullable = true)
 |-- Adventure: long (nullable = true)
 |-- Animation: long (nullable = true)
 |-- Children's: long (nullable = true)
 |-- Comedy: long (nullable = true)
 |-- Crime: long (nullable = true)
 |-- Documentary: long (nullable = true)
 |-- Drama: long (nullable = true)
 |-- Fantasy: long (nullable = true)
 |-- Film-Noir: long (nullable = true)
 |-- Horror: long (nullable = true)
 |-- Musical: long (nullable = true)
 |-- Mystery: long (nullable = true)
 |-- Romance: long (nullable = true)
 |-- Sci-Fi: long (nullable = true)
 |-- Thriller: long (nullable = 

**9. Средняя оценка фильма**

groupBy("item_id") группирует данные по фильму, а agg(avg("rating")) считает среднюю оценку.

In [19]:
avg_rating_by_movie = (
    ratings_spark
    .groupBy("item_id")
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("ratings_count")
    )
    .orderBy(desc("avg_rating"))
)

avg_rating_by_movie.show(20, truncate=False)

+-------+-----------------+-------------+
|item_id|avg_rating       |ratings_count|
+-------+-----------------+-------------+
|1189   |5.0              |3            |
|1599   |5.0              |1            |
|814    |5.0              |1            |
|1536   |5.0              |1            |
|1653   |5.0              |1            |
|1500   |5.0              |2            |
|1293   |5.0              |3            |
|1122   |5.0              |1            |
|1201   |5.0              |1            |
|1467   |5.0              |2            |
|1449   |4.625            |8            |
|119    |4.5              |4            |
|1642   |4.5              |2            |
|1398   |4.5              |2            |
|1594   |4.5              |2            |
|408    |4.491071428571429|112          |
|318    |4.466442953020135|298          |
|169    |4.466101694915254|118          |
|483    |4.45679012345679 |243          |
|114    |4.447761194029851|67           |
+-------+-----------------+-------

**10. Средняя оценка по жанрам**

Жанры в movies_spark закодированы как 0/1. Для каждого жанра считаем среднюю оценку только среди фильмов, где значение жанра равно 1.

Возникнет ошибка из-за col("genre"), заменяем блок создания genre_avg на более простой:

In [21]:
from pyspark.sql.functions import lit

genre_avg_ratings = []

for genre in genre_columns:
    genre_avg = (
        ratings_spark
        .join(movies_spark, on="item_id", how="inner")
        .filter(col(genre) == 1)
        .agg(avg("rating").alias("avg_rating"))
        .withColumn("genre", lit(genre))
    )
    genre_avg_ratings.append(genre_avg)

from functools import reduce
from pyspark.sql import DataFrame

genre_ratings_df = reduce(DataFrame.unionByName, genre_avg_ratings)
genre_ratings_df.orderBy(desc("avg_rating")).show(25, truncate=False)

+------------------+-----------+
|avg_rating        |genre      |
+------------------+-----------+
|3.9215233698788228|Film-Noir  |
|3.815811874866993 |War        |
|3.6873793708484772|Drama      |
|3.6728232189973613|Documentary|
|3.63813155386082  |Mystery    |
|3.6322780881440098|Crime      |
|3.621704948358255 |Romance    |
|3.6132686084142396|Western    |
|3.5766990291262135|Animation  |
|3.5607227022780834|Sci-Fi     |
|3.521396851029471 |Musical    |
|3.5090069495245064|Thriller   |
|3.503526503308369 |Adventure  |
|3.480245417953027 |Action     |
|3.3940734781442745|Comedy     |
|3.3532442216652742|Children's |
|3.2903893172841827|Horror     |
|3.2152366863905324|Fantasy    |
|3.2               |unknown    |
+------------------+-----------+



**11. Популярные и непопулярные фильмы**

Выберем популярность по количеству оценок: чем больше оценок, тем популярнее фильм.

In [22]:
movie_popularity = (
    ratings_spark
    .groupBy("item_id")
    .agg(
        count("rating").alias("ratings_count"),
        avg("rating").alias("avg_rating")
    )
)

# 5 самых популярных фильмов
top_5_popular = (
    movie_popularity
    .orderBy(desc("ratings_count"))
    .limit(5)
)

# 5 самых непопулярных фильмов
top_5_unpopular = (
    movie_popularity
    .orderBy(asc("ratings_count"))
    .limit(5)
)

top_5_popular.show(truncate=False)
top_5_unpopular.show(truncate=False)

+-------+-------------+------------------+
|item_id|ratings_count|avg_rating        |
+-------+-------------+------------------+
|50     |583          |4.3584905660377355|
|258    |509          |3.8035363457760316|
|100    |508          |4.155511811023622 |
|181    |507          |4.007889546351085 |
|294    |485          |3.156701030927835 |
+-------+-------------+------------------+

+-------+-------------+----------+
|item_id|ratings_count|avg_rating|
+-------+-------------+----------+
|1526   |1            |4.0       |
|1458   |1            |4.0       |
|1505   |1            |4.0       |
|1599   |1            |5.0       |
|677    |1            |3.0       |
+-------+-------------+----------+

